# Data Merge & Processing Strategy: System Design Approach

## 🧠 High-Level Design: What Our System Does

**Core Idea**:
1.  **Historical = Ground Truth Operational Schema**: `speed_historic` and `traffic_historic` define our standard schema. We do not modify them.
2.  **Test 1 / 2 / 3 = Different Agency Feeds**: These represent different cities, counties, or vendors sending us new data. Each has its own schema quirks.
3.  **Step 1: Schema Understanding**: We profile the incoming feeds to understand their column maps.
4.  **Step 2: Normalization Views**: We build `norm_*` views to map each feed to the Ground Truth schema (Rename, Cast, Format).
5.  **Step 3: Controlled Merge**: We merge the normalized views with the historical ground truth into a final `operational` table.

This approach ensures that our operational system remains stable while accommodating diverse data sources.

## 🧱 Section 1 — Data Mobilization & Ground Truth Definition

**Goal**: Set up DuckDB, register all raw datasets, and declare historical tables as our **Ground Truth Schema**.
We treat the historical parquet files as the trusted DMV/DoT records. All new data must align to this structure.

In [37]:
import duckdb
import os
import pandas as pd

# 1. Connection Setup
con = duckdb.connect(":memory:")
con.execute("INSTALL httpfs; LOAD httpfs;")

# 2. Register Raw Data Files
DATA_DIR = "../data/opendata"
files = {
    "speed_historic": "nyc_speed_cameras_historic.parquet",
    "speed_test1": "test1_nyc_speed_cameras.json",
    "speed_test2": "test2_nyc_speed_cameras.csv",
    "speed_test3": "test3_nyc_speed_cameras.csv",
    "traffic_historic": "nyc_traffic_violations_historic.parquet",
    "traffic_test1": "test1_nyc_traffic_violations.json",
    "traffic_test2": "test2_nyc_traffic_violations.csv",
    "traffic_test3": "test3_nyc_traffic_violations.csv"
}

print("🚀 Registering Views...")
for name, filename in files.items():
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        if ".parquet" in filename: 
            con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{path}')")
        elif ".json" in filename: 
            con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_json_auto('{path}')")
        elif ".csv" in filename: 
            con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{path}', all_varchar=True)")
        print(f"✅ Registered: {name}")
    else:
        print(f"❌ Missing: {path}")


🚀 Registering Views...
✅ Registered: speed_historic
✅ Registered: speed_test1
✅ Registered: speed_test2
✅ Registered: speed_test3
✅ Registered: traffic_historic
✅ Registered: traffic_test1
✅ Registered: traffic_test2
✅ Registered: traffic_test3


## 🔍 Section 2 — Schema Profiling for New County / Agency Feeds

**Goal**: Before merging, we must inspect the new feeds (`test1`, `test2`, `test3`) to identify schema drift.
This simulates receiving data from different agencies (e.g., Suffolk County, Nassau County) that use different software.

We use `DESCRIBE` and sample queries (`HEAD`) to map the differences.

In [38]:
# Visual Inspection of Raw Feeds
print("\n--- 📸 SPEED CAMERAS (Raw Data Samples) ---")
print("1. Historic (Ground Truth - The Target Schema):")
display(con.sql("SELECT * FROM speed_historic LIMIT 2").df())

print("\n2. Test 1 (JSON - Unnormalized):")
display(con.sql("SELECT * FROM speed_test1 LIMIT 2").df())

print("\n3. Test 2 (CSV - Title Case, Split Dates):")
display(con.sql("SELECT * FROM speed_test2 LIMIT 2").df())

print("\n4. Test 3 (CSV - Abbreviated Columns):")
display(con.sql("SELECT * FROM speed_test3 LIMIT 2").df())



--- 📸 SPEED CAMERAS (Raw Data Samples) ---
1. Historic (Ground Truth - The Target Schema):


,issue_date,created_at,amount_due,county,fine_amount,interest_amount,issuing_agency,judgment_entry_date,license_type,payment_amount,penalty_amount,plate,precinct,reduction_amount,state,summons_number,violation,violation_status,violation_time
0,2025-04-13 08:00:00-04:00,2025-04-20 01:53:02.192000-04:00,0.0,BK,50.0,0.0,DEPARTMENT OF TRANSPORTATION,NaT,PAS,50.0,0.0,HVV8423,0,0.0,NY,4943625630,PHTO SCHOOL ZN SPEED VIOLATION,None,12:06A
1,2023-10-10 08:00:00-04:00,2023-10-16 16:18:20.947000-04:00,0.0,BK,50.0,0.0,DEPARTMENT OF TRANSPORTATION,NaT,PAS,50.0,0.0,D92RJR,0,0.0,NJ,4867038659,PHTO SCHOOL ZN SPEED VIOLATION,None,07:58A



2. Test 1 (JSON - Unnormalized):


,issue_date,created_at,amount_due,county,fine_amount,interest_amount,issuing_agency,judgment_entry_date,license_type,payment_amount,penalty_amount,plate,precinct,reduction_amount,state,summons_number,violation,violation_status,violation_time
0,2025-11-02T09:00:00-05:00,2025-11-09T02:39:46.157-05:00,0.0,QN,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,OMS,50.0,0.0,LWS4720,0,0.0,NY,4970472282,PHTO SCHOOL ZN SPEED VIOLATION,None,02:43P
1,2025-11-03T10:00:00-05:00,2025-11-09T02:39:46.157-05:00,0.0,ST,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,PAS,50.0,0.0,K74KDD,0,0.0,NJ,4970523400,PHTO SCHOOL ZN SPEED VIOLATION,None,07:28P



3. Test 2 (CSV - Title Case, Split Dates):


,Created At,Amount Due,County,Fine Amount,Interest Amount,Issuing Agency,Judgment Entry Date,License Type,Payment Amount,Penalty Amount,Plate,Reduction Amount,State,Summons Number,Violation,Violation Status,Violation Time,Issue Year,Issue Month,Issue Day
0,2025-11-09T02:39:46.157000-0500,0.0,QN,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,OMS,50.0,0.0,LWS4720,0.0,NY,4970472282,PHTO SCHOOL ZN SPEED VIOLATION,None,02:43P,2025,11,2
1,2025-11-09T02:39:46.157000-0500,0.0,ST,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,PAS,50.0,0.0,K74KDD,0.0,NJ,4970523400,PHTO SCHOOL ZN SPEED VIOLATION,None,07:28P,2025,11,3



4. Test 3 (CSV - Abbreviated Columns):


,created_at,amount_due,county,fine_amount,interest_amount,issuing_agency,judgment_entry_date,license_type,payment_amount,penalty_amount,plate,precinct,reduction_amount,state,summons_number,violation,violation_status,violation_time,api_version,issued_date
0,2025-11-09T02:39:46.157000-0500,0.0,QN,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,OMS,50.0,0.0,LWS4720,0,0.0,NY,4970472282,PHTO SCHOOL ZN SPEED VIOLATION,None,02:43P,v2.5_simulated,02-Nov-2025
1,2025-11-09T02:39:46.157000-0500,0.0,ST,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,PAS,50.0,0.0,K74KDD,0,0.0,NJ,4970523400,PHTO SCHOOL ZN SPEED VIOLATION,None,07:28P,v2.5_simulated,03-Nov-2025


In [39]:
# Automated Schema Comparison
def show_features(tables):
    schema_data = []
    for t in tables:
        try:
            df = con.sql(f"DESCRIBE {t}").df()
            for _, row in df.iterrows():
                schema_data.append({"Table": t, "Column": row['column_name'], "Type": row['column_type']})
        except: pass
    
    if not schema_data: return
    # Pivot for clean comparison
    all_features = pd.DataFrame(schema_data)
    pivoted = all_features.pivot_table(index="Column", columns="Table", values="Type", aggfunc='first').fillna("MISSING")
    display(pivoted)

print("--- Schema Differences Detected ---")
show_features([t for t in files.keys() if "speed" in t])

--- Schema Differences Detected ---


Table,speed_historic,speed_test1,speed_test2,speed_test3
Column,,,,
Amount Due,MISSING,MISSING,VARCHAR,MISSING
County,MISSING,MISSING,VARCHAR,MISSING
Created At,MISSING,MISSING,VARCHAR,MISSING
Fine Amount,MISSING,MISSING,VARCHAR,MISSING
Interest Amount,MISSING,MISSING,VARCHAR,MISSING
Issue Day,MISSING,MISSING,VARCHAR,MISSING
Issue Month,MISSING,MISSING,VARCHAR,MISSING
Issue Year,MISSING,MISSING,VARCHAR,MISSING
Issuing Agency,MISSING,MISSING,VARCHAR,MISSING


## 🔄 Section 3 — Build Normalization Views (Per Schema)

**Goal**: Create a translation layer (`VIEW`) for each feed.
We do **not** modify the raw data. Instead, we create a view `norm_speed_test1` that projects the raw data into the **Ground Truth** schema.

**Mapping Strategy**:
- **Dates**: Concatenate Year/Month/Day columns or cast strings to `TIMESTAMP WITH TIME ZONE`.
- **Columns**: Rename 'Title Case' or 'abbrv' to `snake_case`.
- **Types**: Explicitly cast IDs to `BIGINT` and Moneys to `DOUBLE`.

In [40]:
# --------------------------
# SPEED CAMERA NORMALIZATION
# --------------------------

# 1. Test 1: JSON Feed
# Issues: Timestamps are strings. Need casting.
sql_speed_1 = """
SELECT 
    *, 
    try_cast(created_at AS TIMESTAMP WITH TIME ZONE) as created_at_cast,
    try_cast(judgment_entry_date AS TIMESTAMP WITH TIME ZONE) as judgment_entry_date_cast,
    try_cast(issue_date AS TIMESTAMP WITH TIME ZONE) as issue_date_cast
FROM speed_test1
"""
con.execute(f"""CREATE OR REPLACE VIEW norm_speed_test1 AS 
SELECT 
    summons_number,
    plate,
    state,
    license_type,
    issue_date_cast as issue_date,
    violation_time,
    violation,
    judgment_entry_date_cast as judgment_entry_date,
    fine_amount,
    penalty_amount,
    interest_amount,
    reduction_amount,
    payment_amount,
    amount_due,
    precinct,
    county,
    issuing_agency,
    violation_status,
    created_at_cast as created_at
FROM ({sql_speed_1})
""")

# 2. Test 2: CSV Feed (Title Case)
# Issues: 'Issue Year', 'Issue Month', 'Issue Day' split. 'Title Case' names.
con.execute("""CREATE OR REPLACE VIEW norm_speed_test2 AS
SELECT
    "Summons Number" as summons_number,
    "Plate" as plate,
    "State" as state,
    "License Type" as license_type,
    make_timestamp("Issue Year" :: int, "Issue Month" :: int, "Issue Day" :: int, 0, 0, 0) as issue_date, 
    "Violation Time" as violation_time,
    "Violation" as violation,
    try_cast("Judgment Entry Date" AS TIMESTAMP WITH TIME ZONE) as judgment_entry_date,
    "Fine Amount" :: double as fine_amount,
    "Penalty Amount" :: double as penalty_amount,
    "Interest Amount" :: double as interest_amount,
    "Reduction Amount" :: double as reduction_amount,
    "Payment Amount" :: double as payment_amount,
    "Amount Due" :: double as amount_due,
    NULL :: bigint as precinct,
    "County" as county,
    "Issuing Agency" as issuing_agency,
    "Violation Status" as violation_status,
    try_cast("Created At" AS TIMESTAMP WITH TIME ZONE) as created_at
FROM speed_test2
""")

# 3. Test 3: CSV Feed (Abbreviated)
# Issues: 'issued_date' vs 'issue_date'.
con.execute("""CREATE OR REPLACE VIEW norm_speed_test3 AS
SELECT
    summons_number,
    plate,
    state,
    license_type,
    try_cast(issued_date AS TIMESTAMP WITH TIME ZONE) as issue_date,
    violation_time,
    violation,
    NULL :: timestamp with time zone as judgment_entry_date,
    fine_amount :: double as fine_amount,
    penalty_amount :: double as penalty_amount,
    interest_amount :: double as interest_amount,
    reduction_amount :: double as reduction_amount,
    payment_amount :: double as payment_amount,
    amount_due :: double as amount_due,
    precinct :: bigint as precinct,
    county,
    issuing_agency,
    violation_status,
    try_cast(created_at AS TIMESTAMP WITH TIME ZONE) as created_at
FROM speed_test3
""")

print("✅ Speed Camera Normalization Views Created.")

✅ Speed Camera Normalization Views Created.


In [41]:
# --------------------------
# TRAFFIC VIOLATION NORMALIZATION
# --------------------------

# Test 2 (Title Case, Birth Parts)
con.execute("""CREATE OR REPLACE VIEW norm_traffic_test2 AS
SELECT 
    NULL as license_id,
    make_date("Birth Year" :: int, "Birth Month" :: int, 1) as birth_date,
    "Age" :: bigint as age,
    "Violation Code" as violation_code,
    "Violation Year" :: bigint as violation_year,
    "Violation Month" :: bigint as violation_month,
    "Points" :: bigint as points,
    "County" as county
FROM traffic_test2
""")

# Test 3 (Abbreviated)
con.execute("""CREATE OR REPLACE VIEW norm_traffic_test3 AS
SELECT 
    lic_id as license_id,
    try_cast(dob_formatted AS DATE) as birth_date,
    NULL :: bigint as age,
    v_code as violation_code,
    v_year :: bigint as violation_year,
    v_month :: bigint as violation_month,
    points :: bigint as points,
    county
FROM traffic_test3
""")

print("✅ Traffic Violation Normalization Views Created.")

✅ Traffic Violation Normalization Views Created.


## 🧬 Section 4 — Controlled Merge: Build Operational Final Tables

**Goal**: Now that schemas are aligned, we merge them into a single `final` operational table.

**Design Choice**: 
- `speed_historic` is the base.
- We use `UNION ALL BY NAME` to robustly combine fields.
- We use `DISTINCT ON` to handle any overlap (deduplication), prioritizing the latest record.

In [42]:
# Build Final Speed Camera Table
query = """
CREATE OR REPLACE TABLE speed_cameras_final AS 
SELECT DISTINCT ON (summons_number) * 
FROM (
    SELECT * FROM speed_historic                 -- 1. Bases: Historical Ground Truth
    UNION ALL BY NAME
    SELECT * FROM norm_speed_test1               -- 2. Append Normalized Feed 1
    UNION ALL BY NAME
    SELECT * FROM norm_speed_test2               -- 3. Append Normalized Feed 2
    UNION ALL BY NAME
    SELECT * FROM norm_speed_test3               -- 4. Append Normalized Feed 3
)
ORDER BY summons_number, created_at DESC;        -- Deduplication Logic
"""
con.execute(query)
print("🎉 Speed Cameras Final Operational Table Created.")

# Build Final Traffic Violation Table
query_traffic = """
CREATE OR REPLACE TABLE traffic_violations_final AS 
SELECT DISTINCT * 
FROM (
    SELECT * FROM traffic_historic
    UNION ALL BY NAME
    SELECT * FROM traffic_test1
    UNION ALL BY NAME
    SELECT * FROM norm_traffic_test2
    UNION ALL BY NAME
    SELECT * FROM norm_traffic_test3
)
"""
con.execute(query_traffic)
print("🎉 Traffic Violations Final Operational Table Created.")

🎉 Speed Cameras Final Operational Table Created.
🎉 Traffic Violations Final Operational Table Created.


## 🚗 Section 5 — Speed Camera Engine on Top of Final Table

**Goal**: With a unified operational table `speed_cameras_final`, we can now build the Engine.

**The ISA Engine Logic**:
1.  **Windowing**: Filter to last 12 months from `as_of_date`.
2.  **Aggregation**: Group by `plate` + `state`.
3.  **Classification**:
    - 🔴 **TRIGGER**: >= 16 violations
    - 🟡 **WARNING**: 12-15 violations
    - 🟢 **OK**: < 12 violations

In [43]:
# 1. Define Window (12 Months Trailing)
# We simulate 'today' as the latest date in the dataset
res = con.execute("SELECT MAX(issue_date) FROM speed_cameras_final").fetchone()
as_of_date = pd.Timestamp(res[0])
cutoff_date = as_of_date - pd.DateOffset(months=12)

print(f"📅 Engine Run Date: {as_of_date.date()}")
print(f"🔙 Analysis Window: {cutoff_date.date()} to {as_of_date.date()}")

# 2. Run Engine & Generate Summary
engine_sql = f"""
CREATE OR REPLACE TABLE vehicle_speed_summary AS
WITH filtered AS (
    SELECT * FROM speed_cameras_final
    WHERE issue_date >= '{cutoff_date}'
),
aggregated AS (
    SELECT
        plate,
        state,
        COUNT(DISTINCT summons_number) as violations_12m,
        MIN(issue_date) as first_violation_12m,
        MAX(issue_date) as last_violation_12m,
        arg_max(county, issue_date) as county_last_seen,
        SUM(fine_amount) as total_fines_12m
    FROM filtered
    GROUP BY plate, state
)
SELECT
    row_number() OVER (ORDER BY violations_12m DESC, plate) as vehicle_id,
    plate,
    state,
    county_last_seen,
    violations_12m,
    first_violation_12m,
    last_violation_12m,
    total_fines_12m,
    CASE
        WHEN violations_12m >= 16 THEN 'TRIGGER'
        WHEN violations_12m >= 12 THEN 'WARNING'
        ELSE 'OK'
    END as status,
    16 as trigger_threshold,
    12 as warning_lower_bound,
    CAST('{as_of_date}' AS DATE) as as_of_date
FROM aggregated
ORDER BY violations_12m DESC;
"""
con.execute(engine_sql)
print("✅ Speed Camera Engine Complete.")

# 3. Export & Summary
export_path = "../data/exports/vehicle_speed_summary.csv"
os.makedirs("../data/exports", exist_ok=True)
con.execute(f"COPY vehicle_speed_summary TO '{export_path}' (HEADER, DELIMITER ',')")

summary = con.execute("SELECT status, COUNT(*) FROM vehicle_speed_summary GROUP BY status").fetchall()
print(f"\n💾 Results Exported to: {export_path}")
print("📊 Engine Results:")
for s, c in summary: print(f"  {s}: {c}")

print("\n🚨 TOP RISK VEHICLES:")
display(con.sql("SELECT * FROM vehicle_speed_summary WHERE status='TRIGGER' LIMIT 5").df())

📅 Engine Run Date: 2025-10-14
🔙 Analysis Window: 2024-10-14 to 2025-10-14
✅ Speed Camera Engine Complete.

💾 Results Exported to: ../data/exports/vehicle_speed_summary.csv
📊 Engine Results:
  TRIGGER: 76
  OK: 474690

🚨 TOP RISK VEHICLES:


,vehicle_id,plate,state,county_last_seen,violations_12m,first_violation_12m,last_violation_12m,total_fines_12m,status,trigger_threshold,warning_lower_bound,as_of_date
0,1,LCM8254,NY,BK,48,2024-10-15 08:00:00-04:00,2025-06-11 08:00:00-04:00,2400.0,TRIGGER,16,12,2025-10-14
1,2,LHW5598,NY,BX,36,2024-10-21 08:00:00-04:00,2025-10-04 08:00:00-04:00,1800.0,TRIGGER,16,12,2025-10-14
2,3,MHW9481,PA,BK,36,2024-10-14 08:00:00-04:00,2025-02-15 10:00:00-05:00,1800.0,TRIGGER,16,12,2025-10-14
3,4,TLN8692,VA,BK,36,2024-10-16 08:00:00-04:00,2025-10-10 08:00:00-04:00,1800.0,TRIGGER,16,12,2025-10-14
4,5,KXM7078,NY,BX,33,2024-11-24 10:00:00-05:00,2025-10-04 08:00:00-04:00,1650.0,TRIGGER,16,12,2025-10-14
